In [5]:
# If needed, uncomment the line below to install/upgrade TensorFlow
# !pip install -q "tensorflow>=2.15"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


## Task 1
### Loading and preprocessing data

Load the MNIST dataset using `keras.datasets.mnist.load_data()` and prepare the data:
- cast to `float32`;
- normalize pixel values to `[0, 1]` (divide by 255);
- print shapes of train and test arrays.

**Required:** implement `load_mnist()` that returns a tuple `(x_train, y_train, x_test, y_test)` with **normalized** data.

In [6]:
import numpy as np
from tensorflow import keras

def load_mnist():
    """Load and normalize MNIST.
    Returns: x_train, y_train, x_test, y_test.
    """
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    x_train = x_train.astype(np.float32) / 255.0
    x_test = x_test.astype(np.float32) / 255.0

    print("x_train shape:", x_train.shape)
    print("x_test shape:", x_test.shape)

    return x_train, y_train, x_test, y_test

## Task 2
### Basic MLP model in Keras

Implement `build_mlp(input_shape, num_classes)` that creates and returns a **compiled** Keras model.

Architecture requirements (you may add layers, but do not simplify):
- input: `Input(shape=input_shape)`;
- `Flatten()` layer;
- dense layer with **at least 64 units** and `relu` activation;
- `Dropout` layer with rate at least 0.2;
- output: Dense with `num_classes` units and `softmax` activation.

Compilation requirements:
- optimizer: Adam;
- loss: `sparse_categorical_crossentropy`;
- metric: accuracy.

In [8]:
from tensorflow.keras import layers

def build_mlp(input_shape, num_classes):
    """Create and compile a simple MLP model in Keras."""
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

## Task 3
### Training with validation and callbacks

Implement `train_model(model, x_train, y_train, x_val, y_val)` that:
- takes a **compiled** model;
- trains it on `x_train, y_train`;
- uses validation data `x_val, y_val`;
- returns the `history` object.

Requirements:
- `batch_size` not larger than 256;
- at least 5 epochs of training;
- use `EarlyStopping` callback on validation loss with `patience` between 2 and 4 and `restore_best_weights=True`.

In [10]:
def train_model(model, x_train, y_train, x_val, y_val):
    """Train the model and return history."""
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=10,
        batch_size=128,
        callbacks=[early_stopping],
        verbose=1
    )

    return history

## Task 4
### Plotting training curves

Implement `plot_history(history)` that:
- plots loss (train/val) vs epoch;
- plots accuracy (train/val) vs epoch.

Requirements:
- use `matplotlib`;
- add legend and axis labels.


In [12]:
import matplotlib.pyplot as plt

def plot_history(history):
    """Plot training curves from a Keras history object."""
    plt.figure(figsize=(12, 4))

    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss vs Epoch")
    plt.legend()
    plt.grid(True)

    # Accuracy plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history["accuracy"], label="Train Acc")
    plt.plot(history.history["val_accuracy"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Epoch")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

## Task 5
### CNN for MNIST and comparison with MLP

Now you need to:
1. Implement `build_cnn(input_shape, num_classes)` that builds a simple **convolutional** model:
   - at least two convolutional layers (`Conv2D`) with `relu` activation;
   - pooling (e.g. `MaxPooling2D`);
   - `Flatten` layer;
   - one or two dense layers and final `Dense(num_classes, softmax)`.
2. Remember that for a CNN the input shape should be `(H, W, 1)` — you need to add a channel dimension.
3. Train the CNN on the same data as the MLP (you can reuse `train_model`).
4. Compare test accuracy for both models (print both values).

**Required:**
- implement `build_cnn`;
- properly reshape the data;
- train the CNN and print final test accuracy.


In [14]:
def build_cnn(input_shape, num_classes):
    """Create and compile a simple CNN model for MNIST."""
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model